In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

In [2]:
from llama3_2.config import LlamaConfig
from llama3_2.model_trfmrs import LlamaModel
import safetensors.torch as st

vocab_size = 32768
student_config = LlamaConfig(vocab_size=vocab_size)
student_model = LlamaModel(student_config)
student_model = student_model.to(device)


In [ ]:
student_model_states = st.load_file("student_model.safetensors")
student_model.load_state_dict(student_model_states)
student_model

In [ ]:
# freeze student model except embed_tokens
for name, param in student_model.named_parameters():
    if name != "embed_tokens.weight":
        param.requires_grad = False
    else:
        param.requires_grad = True
        print(f"Keeping {name} trainable")


In [ ]:
import pandas as pd
df = pd.read_pickle("token_to_embeddings.pkl")
# sort by token_id
df = df.sort_values(by="token_id")
df

In [ ]:
v = student_model(torch.tensor([[10000]]).to(device))
v.shape

In [23]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, data_df: pd.DataFrame, batch_size: int):
        self.data_df = data_df
        self.batch_size = batch_size
        self.data_size = len(data_df)
        
        # Convert pandas Series to numpy arrays first, then to tensors
        self.input_ids = torch.tensor(data_df['token_id'].values, dtype=torch.long)
        
        # Convert embedding lists to numpy array, then to tensor
        embeddings = np.array(data_df['embedding'].tolist())
        self.targets = torch.tensor(embeddings, dtype=torch.float)

    def __len__(self):
        return self.data_size
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.targets[idx]

def create_dataloader(data_df: pd.DataFrame, batch_size: int):
    dataset = CustomDataset(data_df, batch_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

In [ ]:
train_dataloader = create_dataloader(df, batch_size=256)
len(train_dataloader)

In [69]:
import torch.nn as nn

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)

In [ ]:
import tqdm

epochs = 100

student_model.train()

for epoch in range(epochs):    
    total_loss = 0
    progress_bar = tqdm.tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    
    for batch in progress_bar:
        input_ids, targets = batch
        input_ids = input_ids.to(device)
        targets = targets.to(device)
        v = student_model(input_ids.unsqueeze(0))
        optimizer.zero_grad()
        loss = loss_fn(v.squeeze(0), targets)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        # Update progress bar with current batch loss and average loss
        avg_loss = total_loss / (progress_bar.n + 1)  # Calculate running average
        progress_bar.set_postfix({'batch loss': f'{loss.item():.4f}', 'avg loss': f'{avg_loss:.4f}'})
    
    if (epoch + 1) % 50 == 0:
        # save embedding
        torch.save(student_model.state_dict(), f"student_model_{epoch}_{avg_loss:.4f}.pth")
    
    # Calculate final average loss for the epoch
    avg_loss = total_loss / len(train_dataloader)